first make a env for the projects and install all the requirements

make a backend folder

make a virtaul environment to install all the backend packages

In [ ]:
# for windows
# python -m venv myenv
# for activate
# myenv\scripts\activate

for windows
python -m venv myenv
f

In [ ]:
for mac
# 2. Create virtual environment
python3 -m venv venv

# 3. Activate it
source venv/bin/activate


make a .env file and use a api key

In [ ]:
pip install -r requirements.txt # package installation code

now make a rag_engine.py and write all the rag logic

In [ ]:
#import statemets

In [ ]:
import os
import threading
from pathlib import Path
from dotenv import load_dotenv

# --- INITIALIZATION ---
load_dotenv()

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever

# 2026 Specific: Legacy components moved to langchain_classic
from langchain_classic.retrievers import EnsembleRetriever
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate


In [ ]:
# path

In [ ]:

# --- CONFIGURATION ---
BASE_DIR = Path(__file__).parent
PDF_PATH = BASE_DIR / "consituition of nepal.pdf"
DB_PATH  = str(BASE_DIR / "chroma_db")

# Singleton pattern globals
_rag_chain = None
_init_lock = threading.Lock()
 

rag logic

In [ ]:

def _build_rag_chain():
    """Builds the hybrid RAG pipeline once and caches it."""
    print("🔄 Initializing Hybrid RAG Engine...")

    if not os.getenv("GOOGLE_API_KEY"):
        raise ValueError("GOOGLE_API_KEY is missing from your .env file.")

    # 1. Load PDF and split into chunks
    loader  = PyPDFLoader(str(PDF_PATH))
    docs    = loader.load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
    chunks   = splitter.split_documents(docs)

    # 2. Embeddings (local, CPU-friendly)
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

    # 3. Vector Store (Semantic Search — ChromaDB)
    if os.path.exists(DB_PATH) and os.listdir(DB_PATH):
        print("📁 Loading existing vector store...")
        vectorstore = Chroma(persist_directory=DB_PATH, embedding_function=embeddings)
    else:
        print("✨ Creating new vector store...")
        vectorstore = Chroma.from_documents(
            documents=chunks,
            embedding=embeddings,
            persist_directory=DB_PATH
        )

    vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

    # 4. BM25 Retriever (Keyword Search)
    bm25_retriever   = BM25Retriever.from_documents(chunks)
    bm25_retriever.k = 6

    # 5. Hybrid Ensemble (30% Keywords + 70% Semantic)
    hybrid_retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, vector_retriever],
        weights=[0.3, 0.7]
    )

    # 6. LLM
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.4)

    # 7. Prompt
    system_prompt = (
        "You are an expert legal assistant specializing in the Constitution of Nepal (2015). "
        "Your job is to provide detailed, accurate, and well-structured answers based ONLY on the provided context from the constitution document.\n\n"
        "Guidelines:\n"
        "- Give thorough, comprehensive answers — do NOT give one-liners or vague summaries.\n"
        "- Cite the relevant Article or Part number from the constitution wherever possible (e.g., 'According to Article 18...').\n"
        "- If the question covers multiple aspects, use bullet points or numbered lists to organize your answer clearly.\n"
        "- If the context does not contain enough information to answer the question, clearly state: "
        "'The provided context does not cover this topic sufficiently.'\n"
        "- Never fabricate or guess information not present in the context.\n\n"
        "Context from the Constitution of Nepal (2015):\n{context}"
    )

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{input}"),
    ])

    # 8. Assemble chain
    combine_docs_chain = create_stuff_documents_chain(llm, prompt)
    print("✅ Hybrid RAG Pipeline ready.")
    return create_retrieval_chain(hybrid_retriever, combine_docs_chain)

In [ ]:

def get_answer(query: str) -> str:
    """Thread-safe public function to get answers."""
    global _rag_chain

    if _rag_chain is None:
        with _init_lock:
            if _rag_chain is None:
                _rag_chain = _build_rag_chain()

    result = _rag_chain.invoke({"input": query})
    return result.get("answer", "I encountered an error processing that request.")


# --- TEST ---
if __name__ == "__main__":
    response = get_answer("What is the summary of the fundamental rights?")
    print(f"\nAnswer: {response}")

now create a main.py file and make a backend and also import rag_engine.py and run on server

In [ ]:
"""
main.py
=======
FastAPI server for Nepal Constitution Chatbot.

Endpoints:
  GET  /          → health check
  POST /chat      → accepts { "message": "..." }, returns { "answer": "..." }

Run with:
  uvicorn main:app --reload --port 8000
"""

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from rag_engine import get_answer

# ── App Setup ─────────────────────────────────────────────────────────────────
app = FastAPI(
    title="Nepal Constitution Chatbot API",
    description="RAG-powered chatbot for the Constitution of Nepal (2015)",
    version="1.0.0",
)

# ── CORS — allow the React dev server to talk to this API ─────────────────────
app.add_middleware(
    CORSMiddleware,
    allow_origins=[
        "http://localhost:5173",   # Vite dev server
        "http://localhost:3000",   # CRA dev server (fallback)
    ],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


# ── Request / Response Models ──────────────────────────────────────────────────
class ChatRequest(BaseModel):
    """JSON body sent by the frontend: { "message": "user question here" }"""
    message: str


class ChatResponse(BaseModel):
    """JSON response returned to the frontend: { "answer": "bot reply here" }"""
    answer: str


# ── Routes ────────────────────────────────────────────────────────────────────
@app.get("/", tags=["Health"])
def root():
    """Health check — confirms the API is running."""
    return {"status": "ok", "service": "Nepal Constitution Chatbot API"}


@app.post("/chat", response_model=ChatResponse, tags=["Chat"])
async def chat(request: ChatRequest):
    """
    Main chat endpoint.
    Receives a user question, runs it through the RAG pipeline,
    and returns the AI-generated answer.
    """
    if not request.message.strip():
        raise HTTPException(status_code=400, detail="Message cannot be empty.")

    try:
        answer = get_answer(request.message.strip())
        return ChatResponse(answer=answer)
    except FileNotFoundError as e:
        raise HTTPException(status_code=500, detail=str(e))
    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=f"An error occurred while processing your request: {str(e)}"
        )

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8001)

now connect backend and frontend through api call

In [ ]:
http://localhost:8000/chat